# 📐 Simulasi Tempat Kedudukan Akar (*Root Locus*)
### Sistem Kontrol — Teknik Elektro

Notebook ini menyediakan **fungsi-fungsi simulasi lengkap** untuk analisis
Tempat Kedudukan Akar berikut contoh penggunaan interaktif.
Jalankan sel dari atas ke bawah secara berurutan.

---
**Referensi:** Ogata K., *Modern Control Engineering*, 5th ed., Prentice Hall.
**Library:** [`python-control`](https://python-control.readthedocs.io/)


In [ ]:
# ── Instalasi (hanya perlu dijalankan sekali di Colab) ────────────────
%pip install -q control

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import control
from control import tf, feedback, step_response
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.dpi': 100,
})

# Palet warna konsisten
C1, C2, C3, C4 = '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'

print("✓ Library siap digunakan.")


---
## Bagian 1 — Fungsi Utilitas

Dua fungsi dasar yang dipakai di seluruh notebook:
- **`cetak_info_sistem()`** — menampilkan poles, zeros, asimtot
- **`hitung_parameter_respons()`** — menghitung rise time, overshoot, settling time, ess


In [ ]:
def cetak_info_sistem(sys_tf, nama="Sistem"):
    """Cetak informasi lengkap sebuah fungsi alih."""
    poles = control.poles(sys_tf)
    zeros = control.zeros(sys_tf)
    n, m = len(poles), len(zeros)
    print(f"\n{'='*52}")
    print(f"  INFO SISTEM: {nama}")
    print(f"{'='*52}")
    print(f"  Poles  : {np.round(poles, 4)}")
    print(f"  Zeros  : {np.round(zeros, 4) if m > 0 else 'Tidak ada'}")
    print(f"  Orde   : {n}   |   Jumlah asimtot: {n - m}")
    if n > m:
        sudut = [(2*k+1)*180/(n-m) for k in range(n-m)]
        centroid = (sum(poles.real) - sum(zeros.real)) / (n-m)
        print(f"  Sudut asimtot (°) : {[round(s,1) for s in sudut]}")
        print(f"  Centroid asimtot  : {round(centroid.real, 4)}")
    print(f"{'='*52}")


In [ ]:
def hitung_parameter_respons(t, y, nama="", tampilkan=True):
    """Hitung dan (opsional) cetak parameter respons transien."""
    y_ss = y[-1]
    # Rise time 10→90%
    i10 = np.where(y >= 0.1 * y_ss)[0]
    i90 = np.where(y >= 0.9 * y_ss)[0]
    rt = float(t[i90[0]] - t[i10[0]]) if (len(i10) and len(i90)) else None
    # Peak
    ip = np.argmax(y)
    os_pct = max(0, (y[ip] - y_ss) / y_ss * 100) if y_ss else 0
    # Settling time ±2%
    luar = np.where((y > y_ss*1.02) | (y < y_ss*0.98))[0]
    ts = float(t[luar[-1]]) if len(luar) else float(t[0])

    par = {
        'rise_time':   round(rt, 4) if rt else None,
        'peak_time':   round(float(t[ip]), 4),
        'overshoot':   round(os_pct, 2),
        'settling_time': round(ts, 4),
        'steady_state':  round(float(y_ss), 4),
        'ess':           round(abs(1.0 - float(y_ss)), 4),
    }
    if tampilkan and nama:
        print(f"\n  Parameter Transien [{nama}]")
        print(f"    Rise time     : {par['rise_time']} s")
        print(f"    Peak time     : {par['peak_time']} s")
        print(f"    % Overshoot   : {par['overshoot']} %")
        print(f"    Settling time : {par['settling_time']} s  (±2%)")
        print(f"    Steady-state  : {par['steady_state']}")
        print(f"    Error SS      : {par['ess']}")
    return par

print("✓ Fungsi utilitas siap.")


---
## Bagian 2 — Fungsi Plot

Tiga fungsi visualisasi utama:

| Fungsi | Kegunaan |
|--------|----------|
| `plot_root_locus(G, ...)` | Root locus dengan anotasi poles, zeros, asimtot |
| `plot_respons_step_variasi_K(G, nilai_K, ...)` | Overlay respons step beberapa nilai K |
| `plot_dashboard_lengkap(G, K, ...)` | Dashboard 4-panel: RL + Step + Bode + Tabel |


In [ ]:
def plot_root_locus(sys_tf, judul="Tempat Kedudukan Akar",
                   K_tandai=None, ax=None):
    """Plot root locus dengan anotasi poles, zeros, dan nilai K tertentu."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(8, 6))

    rlist, _ = control.root_locus(sys_tf, plot=False)
    for i in range(rlist.shape[1]):
        ax.plot(rlist[:, i].real, rlist[:, i].imag, color=C1, lw=2, alpha=0.85)
        ax.plot(rlist[:, i].real, -rlist[:, i].imag, color=C1, lw=2, alpha=0.85)

    poles = control.poles(sys_tf)
    zeros = control.zeros(sys_tf)
    ax.plot(poles.real, poles.imag, 'x', color=C4, ms=12, mew=2.5, label='Poles', zorder=5)
    if len(zeros):
        ax.plot(zeros.real, zeros.imag, 'o', color=C3, ms=10, mew=2.5,
                mfc='none', label='Zeros', zorder=5)

    for p in poles:
        ax.annotate(f'  {p:.2f}', xy=(p.real, p.imag), fontsize=8, color=C4)
    for z in zeros:
        ax.annotate(f'  {z:.2f}', xy=(z.real, z.imag), fontsize=8, color=C3)

    if K_tandai:
        for Kv in K_tandai:
            cp = control.poles(feedback(Kv * sys_tf, 1))
            ax.plot(cp.real, cp.imag, 's', color=C2, ms=9, zorder=6,
                    label=f'K={Kv}')
            for p in cp:
                ax.annotate(f' K={Kv}\n({p.real:.2f})', xy=(p.real, p.imag),
                            fontsize=7, color=C2)

    ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.5)
    ax.axvline(0, color=C4, lw=1.5, ls='-.', alpha=0.4, label='Batas stabil')
    ax.set_xlabel('Re(s)', fontsize=10)
    ax.set_ylabel('Im(s)', fontsize=10)
    ax.set_title(judul, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3, ls=':')

    if standalone:
        plt.tight_layout()
        plt.show()
        return fig, ax


In [ ]:
def plot_respons_step_variasi_K(sys_tf, nilai_K, t_max=20,
                                judul="Respons Step — Variasi K"):
    """Overlay respons step loop-tertutup untuk beberapa nilai K."""
    t = np.linspace(0, t_max, 2000)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(nilai_K)))

    for K, warna in zip(nilai_K, colors):
        cl = feedback(K * sys_tf, 1)
        t_o, y_o = step_response(cl, T=t)
        ax.plot(t_o, y_o, color=warna, lw=2, label=f'K = {K}')

    ax.axhline(1,    color='gray', lw=1.2, ls='--', alpha=0.7, label='r(t)=1')
    ax.axhline(1.02, color='red',  lw=0.8, ls=':',  alpha=0.5)
    ax.axhline(0.98, color='red',  lw=0.8, ls=':',  alpha=0.5, label='±2% band')
    ax.set_xlabel('Waktu (s)', fontsize=10)
    ax.set_ylabel('y(t)', fontsize=10)
    ax.set_title(judul, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8.5, loc='upper right')
    ax.grid(True, alpha=0.3, ls=':')
    ax.set_xlim(0, t_max)
    plt.tight_layout()
    plt.show()
    return fig, ax


In [ ]:
def plot_dashboard_lengkap(sys_tf, K_desain, nama_sistem="Sistem",
                           t_max=20, nilai_K_banding=None):
    """Dashboard 4-panel: Root Locus | Respons Step | Bode Mag | Bode Phase + Tabel."""
    fig = plt.figure(figsize=(15, 9))
    fig.suptitle(f'Analisis Lengkap: {nama_sistem}', fontsize=13, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)
    ax_rl   = fig.add_subplot(gs[0, 0])
    ax_st   = fig.add_subplot(gs[0, 1:])
    ax_mag  = fig.add_subplot(gs[1, 0])
    ax_ph   = fig.add_subplot(gs[1, 1])
    ax_tab  = fig.add_subplot(gs[1, 2])

    # ── Root Locus ──────────────────────────────────────────────────
    rlist, _ = control.root_locus(sys_tf, plot=False)
    for i in range(rlist.shape[1]):
        ax_rl.plot(rlist[:, i].real,  rlist[:, i].imag,  color=C1, lw=1.8)
        ax_rl.plot(rlist[:, i].real, -rlist[:, i].imag,  color=C1, lw=1.8)
    poles = control.poles(sys_tf)
    zeros = control.zeros(sys_tf)
    ax_rl.plot(poles.real, poles.imag, 'x', color=C4, ms=11, mew=2.5, label='Poles')
    if len(zeros):
        ax_rl.plot(zeros.real, zeros.imag, 'o', color=C3, ms=9, mew=2.5,
                   mfc='none', label='Zeros')
    cp_K = control.poles(feedback(K_desain * sys_tf, 1))
    ax_rl.plot(cp_K.real, cp_K.imag, 's', color=C2, ms=9, label=f'K={K_desain}', zorder=5)
    ax_rl.axvline(0, color='red', lw=1, ls='-.', alpha=0.4)
    ax_rl.axhline(0, color='gray', lw=0.7, ls='--', alpha=0.4)
    ax_rl.set_title('Root Locus', fontsize=10, fontweight='bold')
    ax_rl.set_xlabel('Re(s)', fontsize=8); ax_rl.set_ylabel('Im(s)', fontsize=8)
    ax_rl.legend(fontsize=7); ax_rl.grid(True, alpha=0.3, ls=':')

    # ── Respons Step ────────────────────────────────────────────────
    t = np.linspace(0, t_max, 2000)
    semua_K = [K_desain] + (nilai_K_banding or [])
    colors = plt.cm.tab10(np.linspace(0, 0.6, len(semua_K)))
    for Kv, warna in zip(semua_K, colors):
        cl = feedback(Kv * sys_tf, 1)
        t_o, y_o = step_response(cl, T=t)
        lw = 2.5 if Kv == K_desain else 1.5
        ls = '-'  if Kv == K_desain else '--'
        label = f'K={Kv}' + (' ← desain' if Kv == K_desain else '')
        ax_st.plot(t_o, y_o, color=warna, lw=lw, ls=ls, label=label)
    ax_st.axhline(1,    color='gray', lw=1.2, ls='--', alpha=0.7, label='r=1')
    ax_st.axhline(1.02, color='red',  lw=0.8, ls=':',  alpha=0.4)
    ax_st.axhline(0.98, color='red',  lw=0.8, ls=':',  alpha=0.4)
    ax_st.set_title('Respons Step Loop-Tertutup', fontsize=10, fontweight='bold')
    ax_st.set_xlabel('Waktu (s)', fontsize=8); ax_st.set_ylabel('y(t)', fontsize=8)
    ax_st.legend(fontsize=7.5, loc='upper right')
    ax_st.grid(True, alpha=0.3, ls=':'); ax_st.set_xlim(0, t_max)

    # ── Bode ────────────────────────────────────────────────────────
    omega = np.logspace(-2, 2, 500)
    try:
        mag, phase, om = control.bode(K_desain * sys_tf, omega, plot=False, dB=True)
        ax_mag.semilogx(om, 20*np.log10(mag + 1e-12), color=C1, lw=2)
        ax_mag.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.6)
        ax_mag.set_title('Bode — Magnitudo', fontsize=10, fontweight='bold')
        ax_mag.set_xlabel('ω (rad/s)', fontsize=8); ax_mag.set_ylabel('|G| (dB)', fontsize=8)
        ax_mag.grid(True, which='both', alpha=0.3, ls=':')

        ax_ph.semilogx(om, np.degrees(phase), color=C2, lw=2)
        ax_ph.axhline(-180, color='red', lw=0.8, ls='--', alpha=0.6, label='-180°')
        ax_ph.set_title('Bode — Fase', fontsize=10, fontweight='bold')
        ax_ph.set_xlabel('ω (rad/s)', fontsize=8); ax_ph.set_ylabel('∠G (°)', fontsize=8)
        ax_ph.legend(fontsize=8); ax_ph.grid(True, which='both', alpha=0.3, ls=':')
    except Exception:
        for a in (ax_mag, ax_ph):
            a.text(0.5, 0.5, 'Bode tidak tersedia', ha='center', va='center',
                   transform=a.transAxes)

    # ── Tabel Parameter ─────────────────────────────────────────────
    cl_d = feedback(K_desain * sys_tf, 1)
    t_o, y_o = step_response(cl_d, T=t)
    par = hitung_parameter_respons(t_o, y_o, tampilkan=False)
    try:
        gm, pm, _, _ = control.margin(K_desain * sys_tf)
        gm_db = f"{20*np.log10(gm):.1f}" if (gm and gm > 0) else '∞'
        pm_deg = f"{pm:.1f}" if pm else '—'
    except Exception:
        gm_db, pm_deg = '—', '—'

    rows = [
        ['K desain', str(K_desain), '—'],
        ['Rise time', str(par['rise_time']), 's'],
        ['% Overshoot', str(par['overshoot']), '%'],
        ['Settling time', str(par['settling_time']), 's'],
        ['Error SS', str(par['ess']), '—'],
        ['Gain Margin', gm_db, 'dB'],
        ['Phase Margin', pm_deg, '°'],
    ]
    ax_tab.axis('off')
    tbl = ax_tab.table(cellText=rows, colLabels=['Parameter', 'Nilai', 'Satuan'],
                       loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.1, 1.6)
    for j in range(3):
        tbl[(0, j)].set_facecolor('#2c3e50')
        tbl[(0, j)].set_text_props(color='white', fontweight='bold')
    ax_tab.set_title('Ringkasan Parameter', fontsize=10, fontweight='bold')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    return fig

print("✓ Fungsi plot siap.")


---
## Demo 1 — Sistem Orde-2: G(s) = 1 / [s(s+2)]

**Sistem:** Kendali posisi sederhana.
Kita akan melihat root locus, menandai beberapa nilai K, dan membandingkan respons.


In [ ]:
# Definisi sistem
G_demo1 = tf([1], [1, 2, 0])   # 1 / (s^2 + 2s) = 1 / [s(s+2)]
cetak_info_sistem(G_demo1, "Demo 1 — G(s) = 1/[s(s+2)]")


In [ ]:
# Root locus dengan K ditandai
plot_root_locus(G_demo1,
                judul="Root Locus — G(s) = 1/[s(s+2)]",
                K_tandai=[0.5, 1, 2])


In [ ]:
# Respons step variasi K
plot_respons_step_variasi_K(G_demo1,
                            nilai_K=[0.5, 1, 2, 5, 10],
                            t_max=12,
                            judul="Respons Step — G(s) = 1/[s(s+2)]")


In [ ]:
# Parameter transien untuk K = 1
cl1 = feedback(1 * G_demo1, 1)
t1, y1 = step_response(cl1, T=np.linspace(0, 12, 2000))
_ = hitung_parameter_respons(t1, y1, nama="K=1, G=1/[s(s+2)]")


---
## Demo 2 — Sistem Orde-3: G(s) = 1 / [s(s+1)(s+3)]

Sistem orde-3 memiliki 3 asimtot.
Asimtot di sudut **±60°** akhirnya melewati kuadran kanan → sistem **bisa tidak stabil** untuk K besar.


In [ ]:
G_demo2 = tf([1], [1, 4, 3, 0])   # 1 / [s(s+1)(s+3)]
cetak_info_sistem(G_demo2, "Demo 2 — G(s) = 1/[s(s+1)(s+3)]")


In [ ]:
plot_dashboard_lengkap(G_demo2,
                       K_desain=3,
                       nama_sistem="G(s) = 1/[s(s+1)(s+3)]",
                       t_max=25,
                       nilai_K_banding=[1, 12])


---
## Demo 3 — Efek Penambahan Zero

Menambahkan zero ke fungsi alih **menarik cabang root locus ke kiri** → memperbaiki kestabilan.

| Sistem | Fungsi Alih |
|--------|-------------|
| Tanpa zero | G(s) = 1/[s(s+4)(s+5)] |
| Zero di s=−2 | G(s) = (s+2)/[s(s+4)(s+5)] |


In [ ]:
G_no_zero  = tf([1],    [1, 9, 20, 0])
G_with_zero = tf([1, 2], [1, 9, 20, 0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Efek Penambahan Zero pada Root Locus', fontsize=12, fontweight='bold')

for ax, G, judul in zip(axes,
                         [G_no_zero, G_with_zero],
                         ['Tanpa Zero\nG(s)=1/[s(s+4)(s+5)]',
                          'Zero di s=−2\nG(s)=(s+2)/[s(s+4)(s+5)]']):
    rlist, _ = control.root_locus(G, plot=False)
    for i in range(rlist.shape[1]):
        ax.plot(rlist[:, i].real,  rlist[:, i].imag,  color=C1, lw=2)
        ax.plot(rlist[:, i].real, -rlist[:, i].imag, color=C1, lw=2)
    p = control.poles(G); z = control.zeros(G)
    ax.plot(p.real, p.imag, 'x', color=C4, ms=12, mew=2.5, label='Poles')
    if len(z):
        ax.plot(z.real, z.imag, 'o', color=C3, ms=10, mew=2.5, mfc='none', label='Zeros')
    ax.axvline(0, color='red', lw=1.2, ls='-.', alpha=0.5, label='Batas stabil')
    ax.axhline(0, color='gray', lw=0.7, ls='--', alpha=0.4)
    ax.set_title(judul, fontsize=10, fontweight='bold')
    ax.set_xlabel('Re(s)'); ax.set_ylabel('Im(s)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, ls=':')

plt.tight_layout()
plt.show()


---
## Demo 4 — Pencarian K Kritis (Batas Kestabilan)

K kritis adalah nilai penguatan terbesar yang masih membuat sistem stabil.
Kita cari dengan **scanning** nilai K dan memeriksa bagian real maksimum poles loop-tertutup.


In [ ]:
G_demo4 = tf([1], [1, 4, 3, 0])   # G(s) = 1/[s(s+1)(s+3)]

K_scan = np.linspace(0.01, 20, 600)
max_real = []
for K in K_scan:
    p = control.poles(feedback(K * G_demo4, 1))
    max_real.append(max(p.real))

max_real = np.array(max_real)
idx_k = np.where(np.diff(np.sign(max_real)))[0]
K_kritis = float(K_scan[idx_k[0]]) if len(idx_k) else None
print(f"K kritis ≈ {K_kritis:.3f}" if K_kritis else "Tidak ditemukan")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Analisis K Kritis — G(s) = 1/[s(s+1)(s+3)]',
             fontsize=12, fontweight='bold')

# Root locus + titik kritis
rlist, _ = control.root_locus(G_demo4, plot=False)
for i in range(rlist.shape[1]):
    ax1.plot(rlist[:, i].real,  rlist[:, i].imag, color=C1, lw=2)
    ax1.plot(rlist[:, i].real, -rlist[:, i].imag, color=C1, lw=2)
ax1.plot(control.poles(G_demo4).real, control.poles(G_demo4).imag,
         'x', color=C4, ms=12, mew=2.5, label='Poles OL')
if K_kritis:
    cp_k = control.poles(feedback(K_kritis * G_demo4, 1))
    ax1.plot(cp_k.real, cp_k.imag, 's', color='purple', ms=10,
             label=f'K_kritis≈{K_kritis:.2f}', zorder=6)
ax1.axvline(0, color='red', lw=1.5, ls='-.', alpha=0.5)
ax1.axhline(0, color='gray', lw=0.7, ls='--', alpha=0.4)
ax1.set_title('Root Locus & Titik Kritis', fontsize=10, fontweight='bold')
ax1.set_xlabel('Re(s)'); ax1.set_ylabel('Im(s)')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3, ls=':')

# Re_max poles vs K
ax2.plot(K_scan, max_real, color=C1, lw=2)
ax2.axhline(0, color='red', lw=1.5, ls='--', alpha=0.8, label='Re=0 (batas stabil)')
if K_kritis:
    ax2.axvline(K_kritis, color='purple', lw=1.5, ls='-.', label=f'K≈{K_kritis:.2f}')
ax2.fill_between(K_scan, max_real, 0, where=max_real > 0,
                 alpha=0.15, color='red', label='Tidak stabil')
ax2.fill_between(K_scan, max_real, 0, where=max_real <= 0,
                 alpha=0.12, color='green', label='Stabil')
ax2.set_title('max[Re(poles CL)] vs K', fontsize=10, fontweight='bold')
ax2.set_xlabel('Penguatan K'); ax2.set_ylabel('max[Re(poles)]')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3, ls=':')

plt.tight_layout()
plt.show()


---
## Fungsi Serba Guna untuk Eksplorasi Mandiri

Gunakan `analisis_cepat()` untuk langsung menganalisis sistem apapun.


In [ ]:
def analisis_cepat(num, den, K_desain, nama="Sistem", t_max=20):
    """
    Fungsi all-in-one: masukkan num/den G(s) dan K,
    dapatkan dashboard lengkap + parameter transien.

    Contoh:
        analisis_cepat([1], [1, 6, 11, 6, 0], K_desain=10, nama='Orde-4')
    """
    G = tf(num, den)
    cetak_info_sistem(G, nama)
    plot_dashboard_lengkap(G, K_desain, nama_sistem=nama, t_max=t_max)
    cl = feedback(K_desain * G, 1)
    t, y = step_response(cl, T=np.linspace(0, t_max, 2000))
    return hitung_parameter_respons(t, y, nama=f"{nama} (K={K_desain})")

# ── Coba di sini ────────────────────────────────────────────────────────
# analisis_cepat([1], [1, 5, 0], K_desain=4, nama="Contoh Anda")
print("Fungsi analisis_cepat() siap. Uncomment baris di atas untuk mencoba!")
